## Part 0 실습 환경 준비

#### Step 1. 라이브러리 임포트 + CIFAR-10 데이터 로딩


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time

torch.manual_seed(42); np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# CIFAR-10 데이터 로딩 (32x32 컬러 이미지, 10 클래스)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])
train_dataset = datasets.CIFAR10('./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.CIFAR10('./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)
classes = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')
print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")
print(f"Image shape: {train_dataset[0][0].shape}")  # [3, 32, 32]


device: cuda


100%|██████████| 170M/170M [00:19<00:00, 8.90MB/s]


Train: 50000, Test: 10000
Image shape: torch.Size([3, 32, 32])


#### Step 2. 학습/평가 공통 함수 정의


In [5]:
def train_model(model, loader, optimizer, criterion, epochs=5):
    model.train()
    history = []
    for epoch in range(epochs):
        total_loss, correct, total = 0, 0, 0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)
        acc = correct / total * 100
        history.append({'loss': total_loss/total, 'acc': acc})
        print(f"  Epoch {epoch+1}/{epochs}  Loss: {total_loss/total:.4f}  Acc: {acc:.1f}%")
    return history

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)
    return correct / total * 100

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## Part 1 AlexNet 스타일 CNN 구현

#### Step 1. AlexNet 스타일 모델 정의

In [6]:
class AlexNetStyle(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),   # 32x32
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                            # 16x16
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                            # 8x8
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                            # 4x4
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 4 * 4, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, 10),
        )
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

alexnet = AlexNetStyle().to(device)
print(f"AlexNet-style params: {count_params(alexnet):,}")


AlexNet-style params: 35,855,178


#### Step 2. AlexNet 학습 + Dropout 효과 확인

In [7]:
# AlexNet 학습
optimizer = optim.Adam(alexnet.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
print("=== AlexNet 학습 ===")
alex_hist = train_model(alexnet, train_loader, optimizer, criterion, epochs=5)
alex_test_acc = evaluate(alexnet, test_loader)
print(f"\nAlexNet Test Acc: {alex_test_acc:.1f}%")

# Dropout 효과: Train Acc vs Test Acc 비교
train_acc = alex_hist[-1]['acc']
print(f"Train Acc: {train_acc:.1f}% vs Test Acc: {alex_test_acc:.1f}%")
print(f"Gap: {train_acc - alex_test_acc:.1f}%p → Dropout이 과적합을 억제합니다")


=== AlexNet 학습 ===
  Epoch 1/5  Loss: 1.5346  Acc: 42.4%
  Epoch 2/5  Loss: 1.1082  Acc: 60.3%
  Epoch 3/5  Loss: 0.9091  Acc: 67.7%
  Epoch 4/5  Loss: 0.7906  Acc: 72.4%
  Epoch 5/5  Loss: 0.6926  Acc: 76.0%

AlexNet Test Acc: 71.7%
Train Acc: 76.0% vs Test Acc: 71.7%
Gap: 4.3%p → Dropout이 과적합을 억제합니다


## Part 2 VGGNet 스타일 블록 구현

#### Step 1. VGG 블록 함수

In [8]:
def make_vgg_block(in_ch, out_ch, num_convs):
    """VGG 스타일: 3x3 Conv를 num_convs번 반복 + MaxPool"""
    layers = []
    for i in range(num_convs):
        layers.append(nn.Conv2d(in_ch if i == 0 else out_ch, out_ch, 3, padding=1))
        layers.append(nn.ReLU(inplace=True))
    layers.append(nn.MaxPool2d(2, 2))
    return nn.Sequential(*layers)

class VGGStyle(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            make_vgg_block(3, 64, 2),     # 32->16
            make_vgg_block(64, 128, 2),   # 16->8
            make_vgg_block(128, 256, 3),  # 8->4
        )
        self.classifier = nn.Sequential(
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, 10),
        )
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

vgg = VGGStyle().to(device)
print(f"VGG-style params: {count_params(vgg):,}")
# 3x3 필터 3개 = 7x7 필터 1개와 같은 Receptive Field
# 파라미터: 3*(3*3*C*C) = 27C^2 vs 7*7*C*C = 49C^2 -> 약 45% 절감


VGG-style params: 5,941,066


#### Step 2. VGG 학습 + 7x7 단일 필터 모델과 비교

In [9]:
# 비교 모델: 7x7 필터 사용
class LargeKernelNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 7, padding=3), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(64, 128, 7, padding=3), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(128, 256, 7, padding=3), nn.ReLU(), nn.MaxPool2d(2,2),
        )
        self.classifier = nn.Sequential(nn.Linear(256*4*4, 1024),
                                        nn.ReLU(), nn.Dropout(0.5), nn.Linear(1024, 10))
    def forward(self, x):
        x = self.features(x); return self.classifier(x.view(x.size(0),-1))

large_k = LargeKernelNet().to(device)
print(f"VGG-style (3x3 반복): {count_params(vgg):,} params")
print(f"Large-kernel (7x7):   {count_params(large_k):,} params")

# VGG 학습
optimizer = optim.Adam(vgg.parameters(), lr=0.001)
print("\n=== VGG-style 학습 ===")
vgg_hist = train_model(vgg, train_loader, optimizer, criterion, epochs=5)
vgg_test_acc = evaluate(vgg, test_loader)
print(f"VGG Test Acc: {vgg_test_acc:.1f}%")


VGG-style (3x3 반복): 5,941,066 params
Large-kernel (7x7):   6,222,474 params

=== VGG-style 학습 ===
  Epoch 1/5  Loss: 1.6389  Acc: 38.1%
  Epoch 2/5  Loss: 1.1130  Acc: 60.0%
  Epoch 3/5  Loss: 0.8667  Acc: 69.4%
  Epoch 4/5  Loss: 0.7117  Acc: 75.2%
  Epoch 5/5  Loss: 0.5994  Acc: 78.9%
VGG Test Acc: 77.0%


## Part 3 ResNet - Skip Connection 구현

#### Step 1. Residual Block 직접 구현

In [10]:
class ResidualBlock(nn.Module):
    """Basic Residual Block: y = F(x) + x"""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        identity = x                        # Skip Connection: 입력을 저장
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + identity                 # F(x) + x  <- 핵심!
        return F.relu(out)

# 동작 확인
block = ResidualBlock(64)
test_input = torch.randn(1, 64, 8, 8)
test_output = block(test_input)
print(f"Input: {test_input.shape} -> Output: {test_output.shape}")
print("Skip Connection 정상 동작: 입출력 shape 동일!")


Input: torch.Size([1, 64, 8, 8]) -> Output: torch.Size([1, 64, 8, 8])
Skip Connection 정상 동작: 입출력 shape 동일!


#### Step 2. ResNet 모델 정의 + 학습

In [ ]:
class SimpleResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())
        self.layer1 = nn.Sequential(ResidualBlock(64), ResidualBlock(64))
        self.pool1 = nn.MaxPool2d(2, 2)  # 32->16
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU())
        self.layer2 = nn.Sequential(ResidualBlock(128), ResidualBlock(128))
        self.pool2 = nn.MaxPool2d(2, 2)  # 16->8
        self.gap = nn.AdaptiveAvgPool2d(1)   # Global Average Pooling
        self.fc = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.pool1(self.layer1(x))
        x = self.conv2(x)
        x = self.pool2(self.layer2(x))
        x = self.gap(x).view(x.size(0), -1)
        return self.fc(x)

resnet = SimpleResNet().to(device)
print(f"SimpleResNet params: {count_params(resnet):,}")
optimizer = optim.Adam(resnet.parameters(), lr=0.001)
print("\n=== ResNet 학습 ===")
res_hist = train_model(resnet, train_loader, optimizer, criterion, epochs=5)
res_test_acc = evaluate(resnet, test_loader)
print(f"ResNet Test Acc: {res_test_acc:.1f}%")


SimpleResNet params: 816,906

=== ResNet 학습 ===
  Epoch 1/5  Loss: 1.2956  Acc: 52.9%
  Epoch 2/5  Loss: 0.8587  Acc: 69.6%


#### Step 3. Skip Connection 유무 비교 실험

In [ ]:
class PlainBlock(nn.Module):
    """Skip Connection 없는 Plain Block"""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(channels)
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out)   # <- identity 더하기 없음!

class PlainNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())
        self.layer1 = nn.Sequential(PlainBlock(64), PlainBlock(64))
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU())
        self.layer2 = nn.Sequential(PlainBlock(128), PlainBlock(128))
        self.pool2 = nn.MaxPool2d(2, 2)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, 10)
    def forward(self, x):
        x = self.conv1(x); x = self.pool1(self.layer1(x))
        x = self.conv2(x); x = self.pool2(self.layer2(x))
        return self.fc(self.gap(x).view(x.size(0), -1))

plain = PlainNet().to(device)
optimizer_p = optim.Adam(plain.parameters(), lr=0.001)
print("=== PlainNet (Skip 없음) 학습 ===")
plain_hist = train_model(plain, train_loader, optimizer_p, criterion, epochs=5)
plain_test = evaluate(plain, test_loader)
print(f"\n비교: ResNet {res_test_acc:.1f}% vs PlainNet {plain_test:.1f}%")
print("-> Skip Connection이 학습을 안정화시키고 성능을 향상시킵니다!")


## Part 4 Depthwise Separable Cov (MobileNet)

#### Step 1. Standard Conv vs DW Separable Conv 비교

In [ ]:
# Standard Convolution
standard_conv = nn.Conv2d(64, 128, kernel_size=3, padding=1)

# Depthwise Separable Convolution
dw_conv = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64)  # Depthwise
pw_conv = nn.Conv2d(64, 128, kernel_size=1)                        # Pointwise

std_params = sum(p.numel() for p in standard_conv.parameters())
dw_params  = sum(p.numel() for p in dw_conv.parameters())
pw_params  = sum(p.numel() for p in pw_conv.parameters())
sep_params = dw_params + pw_params

print(f"Standard Conv params:  {std_params:,}")
print(f"DW Conv params:        {dw_params:,}")
print(f"PW Conv (1x1) params:  {pw_params:,}")
print(f"DW+PW total params:    {sep_params:,}")
print(f"\n절감율: {sep_params/std_params*100:.1f}% (약 {std_params/sep_params:.1f}배 절감)")

# 출력 비교
x = torch.randn(1, 64, 32, 32)
out_std = standard_conv(x)
out_sep = pw_conv(dw_conv(x))
print(f"\nStandard output: {out_std.shape}")
print(f"DW+PW output:    {out_sep.shape}  <- 동일한 shape!")


#### Step 2. MobileNet 스타일 경량 모델 구현 + 학습

In [ ]:
class DWSepConv(nn.Module):
    """Depthwise Separable Convolution Block"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, 3, padding=1, groups=in_ch)
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.pw = nn.Conv2d(in_ch, out_ch, 1)
        self.bn2 = nn.BatchNorm2d(out_ch)
    def forward(self, x):
        x = F.relu(self.bn1(self.dw(x)))
        return F.relu(self.bn2(self.pw(x)))

class MobileNetStyle(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            DWSepConv(32, 64),
            nn.MaxPool2d(2, 2),     # 32->16
            DWSepConv(64, 128),
            nn.MaxPool2d(2, 2),     # 16->8
            DWSepConv(128, 256),
            nn.MaxPool2d(2, 2),     # 8->4
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, 10)
    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).view(x.size(0), -1)
        return self.fc(x)

mobile = MobileNetStyle().to(device)
print(f"MobileNet-style params: {count_params(mobile):,}")
optimizer = optim.Adam(mobile.parameters(), lr=0.001)
print("\n=== MobileNet-style 학습 ===")
mob_hist = train_model(mobile, train_loader, optimizer, criterion, epochs=5)
mob_test_acc = evaluate(mobile, test_loader)
print(f"MobileNet Test Acc: {mob_test_acc:.1f}%")


## Part 5 모델 비교 종합 실험

#### Step 1. 4개 모델 성능 비교표 출력

In [ ]:
results = {
    'AlexNet-style':   {'params': count_params(alexnet), 'acc': alex_test_acc},
    'VGG-style':       {'params': count_params(vgg),     'acc': vgg_test_acc},
    'SimpleResNet':    {'params': count_params(resnet),  'acc': res_test_acc},
    'MobileNet-style': {'params': count_params(mobile),  'acc': mob_test_acc},
}

print("=" * 55)
print(f"{'모델':<18} {'파라미터 수':>12} {'Test Acc':>10}")
print("-" * 55)
for name, r in results.items():
    print(f"{name:<18} {r['params']:>12,} {r['acc']:>9.1f}%")
print("=" * 55)
print("\n핵심 관찰:")
print("1. ResNet: Skip Connection으로 깊은 학습 안정화")
print("2. MobileNet: 최소 파라미터로 경쟁력 있는 성능")
print("3. VGG: 3x3 반복으로 큰 Receptive Field 확보")
print("4. AlexNet: Dropout으로 과적합 억제")


#### Step 2. 학습 곡선 시각화 + Feature Map 비교

In [ ]:
# 학습 곡선 비교
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
histories = {'AlexNet': alex_hist, 'VGG': vgg_hist,
             'ResNet': res_hist, 'MobileNet': mob_hist}
for name, h in histories.items():
    axes[0].plot([e['loss'] for e in h], label=name)
    axes[1].plot([e['acc'] for e in h], label=name)
axes[0].set_title('Training Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')
axes[1].set_title('Training Accuracy (%)'); axes[1].legend(); axes[1].set_xlabel('Epoch')
plt.tight_layout(); plt.show()

# Feature Map 시각화 (ResNet 첫 번째 레이어)
img, label = test_dataset[0]
img_input = img.unsqueeze(0).to(device)
with torch.no_grad():
    feat = resnet.conv1(img_input)
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(16):
    axes[i//8, i%8].imshow(feat[0, i].cpu(), cmap='viridis')
    axes[i//8, i%8].axis('off')
    axes[i//8, i%8].set_title(f'Ch {i}', fontsize=8)
fig.suptitle(f'ResNet Conv1 Feature Maps (label: {classes[label]})')
plt.tight_layout(); plt.show()
